In [ ]:
import os
os.chdir(r"D:\Computer Vision\Counting-people-in-a-marathon-using-YOLOv8-main\Counting-people-in-a-marathon-using-YOLOv8-main")
import cv2
from tracker import Tracker
import numpy as np
import pandas as pd
from ultralytics import YOLO
import cvzone

In [ ]:
model=YOLO('yolov8s.pt')

In [ ]:
def points(events, x, y, flags, param):
    if events == cv2.EVENT_MOUSEMOVE:
        point = [x, y]
#         print(point)

        
cv2.namedWindow("points")
cv2.setMouseCallback("points" , points)



cap=cv2.VideoCapture('p3.mp4')


my_file = open("coco.txt", "r")
data = my_file.read()
class_list = data.split("\n") 

count=0

tracker=Tracker()

cy1=383

offset=4

counter=[]
while True:    
    ret,frame = cap.read()
    if not ret:
        break
    frame=cv2.resize(frame,(1020,500))
   

    results=model.predict(frame)
 #   print(results)
#     a=results[0].boxes.boxes
    a = results[0].boxes.xyxy
    px=pd.DataFrame(a).astype("float")
#    print(px)
    list=[]
             
    for index,row in px.iterrows():
#        print(row)
 
        x1=int(row[0])
        y1=int(row[1])
        x2=int(row[2])
        y2=int(row[3])
        d=int(row[5]) if len(row) > 5 else 0  # Class index
        c=class_list[d]
        if 'person' in c:
            list.append([x1,y1,x2,y2])
            
    bbox_id=tracker.update(list)
    for bbox in bbox_id:
        x3,y3,x4,y4,id=bbox
        cx=int(x3+x4)//2
        cy=int(y3+y4)//2
        cv2.putText(frame,str(id),(x3,y3),cv2.FONT_HERSHEY_COMPLEX,0.6,(255,255,255),1)
        cv2.rectangle(frame,(x3,y3),(x4,y4),(255,0,0),2)
      
        if cy1<(cy+offset) and cy1 > (cy-offset):
            if counter.count(id)==0:
                counter.append(id)
                cv2.circle(frame,(cx,cy),4,(0,0,255),-1)
                
    cv2.line(frame ,(335,cy1),(696,cy1),(0,255,0),2)
    u=(len(counter))
    cvzone.putTextRect(frame, f'people count:-{u}', (50,60),2,2)
    cv2.imshow("RGB", frame)
    if cv2.waitKey(1)==ord("q"):
        break
cap.release()
cv2.destroyAllWindows()